# Motivation (real T/T_DAG circuit via tsim)

Mirrors `motivation.ipynb` exactly, but replaces every **S** gate with **T** and every **S_DAG** with **T_DAG** so that the cultivation circuit uses the true non-Clifford magic-state injection.
Sampling is done by **tsim** (ZX stabilizer-rank decomposition), which supports non-Clifford gates.

Key differences from `motivation.ipynb`:
- `TsimCircuitProxy` wraps the noisy circuit so all structural operations (DEM, detector coords, …) still run through stim, while **`compile_detector_sampler()`** is redirected to tsim.
- `DensityHeatMapCollection` uses the DEM sampler (Clifford approximation) → structure should look **identical** to the S-gate notebook.
- `GapSensitivityCollect` uses tsim for circuit-level sampling → gap statistics reflect **true T-gate behavior**.
- `LifeTimeCollect` still uses `stim.FlipSimulator` internally (stim ignores gate tags) → survival curve is an **approximation** (T treated as S for the flip sim).


In [ ]:
import os
os.environ['JAX_PLATFORMS'] = 'cpu'  # remove this line if you have a working CUDA GPU

import pathlib
import re
import sys

# ── project src ──────────────────────────────────────────────────────────────
src_path = pathlib.Path().resolve().parent / 'magic_state_cultivation' / 'upstream' / 'src'
assert src_path.exists(), f'src not found: {src_path}'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# ── tsim (cloned repo) ───────────────────────────────────────────────────────
tsim_src = pathlib.Path().resolve().parent.parent / 'tsim' / 'src'
assert tsim_src.exists(), f'tsim src not found: {tsim_src}'
if str(tsim_src) not in sys.path:
    sys.path.insert(0, str(tsim_src))

import stim
import tsim
import cultiv
import gen

from generate_circuit_with_stage_map import CircuitGenerator
from density_heat_map_collection import DensityHeatMapCollection
from logical_dens_heat_map_collection import LogicalDensityHeatMapCollection
from gap_sensitivity_collect import GapSensitivityCollect
from lifetime_collect import LifeTimeCollect
from Detector_selection_3D import DetectorSelection3D
from gap_threshold_get import GapThreshotGenerate

print(f'tsim {tsim.__version__}  |  stim {stim.__version__}')

In [ ]:
def _tag_s_as_t(circuit: stim.Circuit) -> stim.Circuit:
    """Replace S -> S[T] and S_DAG -> S_DAG[T] at instruction level (handles REPEAT blocks).
    Uses mini circuit string parsing to create tagged instructions, compatible with
    older stim versions that don't support append(..., tag='T').
    """
    new = stim.Circuit()
    for instr in circuit:
        if isinstance(instr, stim.CircuitRepeatBlock):
            new.append(stim.CircuitRepeatBlock(instr.repeat_count, _tag_s_as_t(instr.body_copy())))
        elif instr.name == 'S' and getattr(instr, 'tag', '') == '':
            target_str = ' '.join(str(t.value) for t in instr.targets_copy())
            for ti in stim.Circuit(f'S[T] {target_str}'):
                new.append(ti)
        elif instr.name == 'S_DAG' and getattr(instr, 'tag', '') == '':
            target_str = ' '.join(str(t.value) for t in instr.targets_copy())
            for ti in stim.Circuit(f'S_DAG[T] {target_str}'):
                new.append(ti)
        else:
            new.append(instr)
    return new


class TsimCircuitProxy:
    """Wraps a stim.Circuit (with S[T] / S_DAG[T] tags) so that:
    - All structural operations (DEM, detecting_regions, etc.) delegate to stim.
    - compile_detector_sampler() is redirected to tsim for T-gate simulation.
    """

    def __init__(self, stim_circ: stim.Circuit):
        self._stim = stim_circ

    def __getattr__(self, name):
        return getattr(self._stim, name)

    def __len__(self):
        return len(self._stim)

    def __iter__(self):
        return iter(self._stim)

    def __str__(self):
        return str(self._stim)

    def __repr__(self):
        return f'TsimCircuitProxy({self._stim.num_qubits}q, {self._stim.num_detectors}dets)'

    @property
    def num_qubits(self):      return self._stim.num_qubits
    @property
    def num_detectors(self):   return self._stim.num_detectors
    @property
    def num_measurements(self):return self._stim.num_measurements
    @property
    def num_observables(self): return self._stim.num_observables

    def copy(self) -> 'TsimCircuitProxy':
        return TsimCircuitProxy(self._stim.copy())

    def append(self, name, targets=None, arg=None, **kwargs):
        self._stim.append(name, targets if targets is not None else [], arg)

    def detector_error_model(self, **kwargs) -> stim.DetectorErrorModel:
        tc = tsim.Circuit.from_stim_program(self._stim)
        try:
            return tc.detector_error_model(**kwargs)
        except Exception:
            kwargs.pop('decompose_errors', None)
            return self._stim.detector_error_model(
                allow_gauge_detectors=True,
                approximate_disjoint_errors=True,
                **{k: v for k, v in kwargs.items()
                   if k in ('flatten_loops', 'ignore_decomposition_failures')}
            )

    def _has_t_gates(self) -> bool:
        """Return True if the circuit contains any S[T] / S_DAG[T] (non-Clifford) gates."""
        return any(
            instr.name in ('S', 'S_DAG') and instr.tag == 'T'
            for instr in self._stim.flattened()
        )

    def compile_detector_sampler(self, **kwargs):
        # Fall back to stim for purely Clifford circuits (no T gates).
        # This avoids crashing JAX with huge circuits that have nothing non-Clifford.
        if not self._has_t_gates():
            return self._stim.compile_detector_sampler(**kwargs)
        tc = tsim.Circuit.from_stim_program(self._stim)
        return tc.compile_detector_sampler(**kwargs)


def make_tsim_cg(
    *,
    basis, gateset, circuit_type, noise_model, noise_strength,
    injection_protocol, r_in_escape, d1, r_post_escape, d2,
    HasNoise=True,
) -> CircuitGenerator:
    """Build a CircuitGenerator whose noisy_circuit is a TsimCircuitProxy
    with S / S_DAG tagged as S[T] / S_DAG[T] (= T / T_DAG for tsim).
    """
    cg = CircuitGenerator.from_input_params(
        basis=basis, gateset=gateset,
        circuit_type=circuit_type,
        noise_model=noise_model,
        noise_strength=noise_strength,
        injection_protocol=injection_protocol,
        r_in_escape=r_in_escape, d1=d1,
        r_post_escape=r_post_escape, d2=d2,
        HasNoise=HasNoise,
    )
    cg.generate()
    if cg.noisy_circuit is not None:
        tagged = _tag_s_as_t(cg.noisy_circuit)
        n_tagged = sum(
            1 for instr in cg.noisy_circuit.flattened()
            if instr.name in ('S', 'S_DAG') and getattr(instr, 'tag', '') == ''
        )
        cg.noisy_circuit = TsimCircuitProxy(tagged)
        print(f'd1={d1} d2={d2} gateset={gateset}: tagged {n_tagged} S/S_DAG -> T/T_DAG')
    return cg


## Circuit type constants (same as motivation.ipynb)

In [ ]:
circuit_types = [
    'escape-to-big-matchable-code',
    'idle-matchable-code',
    'surface-code-memory',
    'inject+cultivate',
    'end2end-inplace-distillation',
    'escape-to-big-color-code',
    'surface-code-cnot'
]
injection_protocols = ['degenerate', 'bell', 'unitary']
noise_models = ['circuit-level-SI1000', 'uniform-depolarizing']

## Generate the circuit (density section)
Same parameters as motivation.ipynb cell 5.

In [ ]:
basis             = 'Y'
gateset           = 'css'
circuit_type      = circuit_types[4]          # 'end2end-inplace-distillation'
noise_model       = noise_models[1]            # 'circuit-level-SI1000'
noise_strength    = 0.001
injection_protocol= injection_protocols[2]    # 'unitary'
r_in_escape       = 6
r_post_escape     = 0
d1                = 3
d2                = 7

cg = make_tsim_cg(
    basis=basis, gateset=gateset,
    circuit_type=circuit_type,
    noise_model=noise_model,
    noise_strength=noise_strength,
    injection_protocol=injection_protocol,
    r_in_escape=r_in_escape, d1=d1,
    r_post_escape=r_post_escape, d2=d2,
)

In [ ]:
manual_viz = {
    'injection':  (0, 2),
    'cultivation':(3, 3),
    'escape':     (4, 11),
}

## Error and non-trivial detector density
Uses the DEM sampler → Clifford approximation → **identical to motivation.ipynb** (validates decoding-graph structure is preserved).

In [ ]:
shots_dens = 10000

### All shots

In [ ]:
dens_all = DensityHeatMapCollection(
    circuit_generator=cg,
    apply_postselection=False,
    count_mode='all_shots',
    decompose_errors=False,
)
dens_all.collect(shots=shots_dens)

In [ ]:
dens_all.heatmap_for_notebook(visualization_stage_timeline_map=manual_viz)

In [ ]:
dens_all.detector_histogram_for_notebook()

In [ ]:
dens_all.hyperedge_histogram_for_notebook()

### With post-selection

In [ ]:
dens_select = DensityHeatMapCollection(
    circuit_generator=cg,
    apply_postselection=True,
    count_mode='accepted_only',
    decompose_errors=False,
)
dens_select.collect(shots=shots_dens)

In [ ]:
dens_select.heatmap_for_notebook(visualization_stage_timeline_map=manual_viz)

In [ ]:
dens_select.detector_histogram_for_notebook()

In [ ]:
dens_select.hyperedge_histogram_for_notebook()

## Logical density

In [ ]:
# Logical density uses 'cx' gateset (uniform depolarizing)
cg_cx = make_tsim_cg(
    basis=basis, gateset='css',
    circuit_type=circuit_type,
    noise_model=noise_models[1],
    noise_strength=noise_strength,
    injection_protocol=injection_protocol,
    r_in_escape=r_in_escape, d1=d1,
    r_post_escape=r_post_escape, d2=d2,
)

In [ ]:
shots_logic = 10000

### All shots

In [ ]:
logic_all = LogicalDensityHeatMapCollection(
    circuit_generator=cg_cx,
    apply_postselection=False,
    count_mode='all_shots',
    decompose_errors=False,
)
logic_all.collect(shots=shots_logic)

In [ ]:
display(logic_all.heatmap_for_notebook(visualization_stage_timeline_map=manual_viz))

In [ ]:
display(logic_all.detector_histogram_for_notebook())
display(logic_all.edge_histogram_for_notebook())

### With post-selection

In [ ]:
logic_post = LogicalDensityHeatMapCollection(
    circuit_generator=cg_cx,
    apply_postselection=True,
    count_mode='accepted_only',
    decompose_errors=True,
)
logic_post.collect(shots=shots_logic)

In [ ]:
display(logic_post.heatmap_for_notebook(visualization_stage_timeline_map=manual_viz))

In [ ]:
display(logic_post.detector_histogram_for_notebook())
display(logic_post.edge_histogram_for_notebook())

## Confidence Gap Sensitivity
Gap sensitivity uses tsim's circuit-level sampler → **T-gate effects properly simulated**.

### Distance 3 color code

In [ ]:
cg_gap_d3 = make_tsim_cg(
    basis='Y', gateset='css',
    circuit_type=circuit_types[4],
    noise_model=noise_models[1],
    noise_strength=0.001,
    injection_protocol=injection_protocols[2],
    d1=3, d2=11,
    r_in_escape=3, r_post_escape=10,
)

In [ ]:
gap_col_d3 = GapSensitivityCollect(circuit_generator=cg_gap_d3)

#### Single detector gap sensitivity

In [ ]:
gap_col_d3.collect_single_detector_gap_sens()
gap_col_d3.single_detector_gap_sens_plot()

#### Overall sensitivity

In [ ]:
shots_overall_gap_sens = 100000
gap_col_d3.collect_overall_gap_sens(shots=shots_overall_gap_sens)

##### Gap sensitivity

In [ ]:
gap_col_d3.overall_detector_gap_sens_plot()

##### Error sensitivity

In [ ]:
gap_col_d3.overall_detector_err_sens_plot()

### Distance 5 color code

In [ ]:
cg_gap_d5 = make_tsim_cg(
    basis='Y', gateset='css',
    circuit_type=circuit_types[4],
    noise_model=noise_models[1],
    noise_strength=0.001,
    injection_protocol=injection_protocols[2],
    d1=5, d2=17,
    r_in_escape=5, r_post_escape=10,
)

In [ ]:
gap_col_d5 = GapSensitivityCollect(circuit_generator=cg_gap_d5)

#### Single detector gap sensitivity

In [ ]:
gap_col_d5.collect_single_detector_gap_sens()
gap_col_d5.single_detector_gap_sens_plot()

#### Overall sensitivity

In [ ]:
shots_overall_gap_sens_d5 = 100000
gap_col_d5.collect_overall_gap_sens(shots=shots_overall_gap_sens_d5)

##### Gap sensitivity

In [ ]:
gap_col_d5.overall_detector_gap_sens_plot()

##### Error sensitivity

In [ ]:
gap_col_d5.overall_detector_err_sens_plot()

## Life time
⚠️ `LifeTimeCollect` uses `stim.FlipSimulator` internally, which treats the S[T]-tagged gates as plain S gates.
The survival curve is therefore an **approximation** — it matches the S-gate (`motivation.ipynb`) result.
The gap-based acceptance rate inside `_estimate_success_rate` **does** use tsim (via the proxy's `compile_detector_sampler`).

### Distance 3 color code

In [ ]:
cg_lt_d3 = make_tsim_cg(
    basis='Y', gateset='css',
    circuit_type=circuit_types[4],
    noise_model=noise_models[1],
    noise_strength=0.001,
    injection_protocol=injection_protocols[2],
    d1=3, d2=15,
    r_in_escape=3, r_post_escape=10,
)

In [ ]:
feedback_time_ns        = 200
avg_decoder_latency_ns  = 10000

lt = LifeTimeCollect(
    distance=3,
    circuit_generator=cg_lt_d3,
    feedback_time_ns=feedback_time_ns,
    decoder_latency_ns=avg_decoder_latency_ns,
)
lt.collect(num_shots=1024*100, gap_threshold=None, success_rate_mode='auto_high_rejection')

In [ ]:
print(lt.success_rate)

#### Life

In [ ]:
display(lt.plot_for_notebook())

#### Time

In [ ]:
display(lt.plot_time_for_notebook())

### Distance 5 color code

In [ ]:
cg_lt_d5 = make_tsim_cg(
    basis='Y', gateset='css',
    circuit_type=circuit_types[4],
    noise_model=noise_models[1],
    noise_strength=0.001,
    injection_protocol=injection_protocols[2],
    d1=5, d2=15,
    r_in_escape=5, r_post_escape=10,
)

In [ ]:
lt_5 = LifeTimeCollect(
    distance=5,
    circuit_generator=cg_lt_d5,
    feedback_time_ns=100,
    decoder_latency_ns=5000,
)
lt_5.collect(num_shots=1024*100, gap_threshold=None, success_rate_mode='auto_high_rejection')

In [ ]:
print(lt_5.success_rate)

#### Life

In [ ]:
display(lt_5.plot_for_notebook())

#### Time

In [ ]:
display(lt_5.plot_time_for_notebook())

## Cycle-Accurate Estimator

In [ ]:
import importlib
import runtime_estimator as re_mod
importlib.reload(re_mod)

### Distance 3 color code

In [ ]:
# d = 3 color code

# Generate the circuit
basis = 'Y'
gateset = 'css'
circuit_type = circuit_types[4]  # 'end2end-inplace-distillation'
noise_model = noise_models[1]  # 'circuit-level-SI1000'
noise_strength = 0.001
injection_protocol = injection_protocols[2]  # 'unitary'
r_in_escape = 3   # r1
r_post_escape = 10 # r2
d1 = 3
d2 = 15
HasNoise = True

epoch = 1000000

feedback_time_ns = 2000
manual_avg_decoder_latency_ns = 10000
use_measured_decode_time = True


re = re_mod.RuntimeEstimator(
    epoch=epoch,                    # how many successful outputs to emulate
    basis=basis,
    gateset=gateset,
    circuit_type=circuit_type,
    noise_model=noise_model,
    noise_strength=noise_strength,
    distance=d1,                   # or 5
    d2=d2,
    feedback_time_ns=feedback_time_ns,
    decoder_latency_ns=manual_avg_decoder_latency_ns,  # 10 us
    use_measured_decode_time=use_measured_decode_time,
    wait_rounds_for_plot=r_post_escape,      # plot expansion only
)

In [ ]:
cg = make_tsim_cg(
    basis=basis,
    gateset=gateset,
    circuit_type=circuit_type,
    noise_model=noise_model,
    noise_strength=noise_strength,
    injection_protocol=injection_protocol,
    r_in_escape=r_in_escape,
    d1=d1,
    r_post_escape=r_post_escape,
    d2=d2,
)

selector = DetectorSelection3D(circuit_generator=cg)

# Build mask (rectangle)
left_len = 8
top_len = 7
t = 5
# mode = 'rectangle'   # or 'triangle'
mode = 'triangle'

mask_bool, mask_packed = selector.build_color_region_mask(
    left_len=left_len,
    top_len=top_len,
    t=t,
    mode=mode,   # or 'triangle'
    stage_name='escape'
)

In [ ]:
# Inline visualization in notebook
selector.visualize_selected_region_plot()

##### No selection based on gap threshold in escape stage

In [ ]:
gap_check = False

re.estimate_runtime(gap_check=gap_check, partial_mask=False, selector=selector, left_len=left_len, top_len=top_len, t=t, mode=mode, stage_name="escape", decoder_time_measure_mode="parallel")

In [ ]:
import numpy as np

In [ ]:
print("decode calls:", len(re.decode_call_times_ns))
print("mean decode ns:", np.mean(re.decode_call_times_ns))
print("p50/p90 ns:", np.percentile(re.decode_call_times_ns, [50, 90]))

In [ ]:
display(re.plot_life_time_notebook())
avg_decoding_latency = re.report_average_decode_latency_ns()
logical_error_rate = re.logical_error_rate
# print(f"Logical error rate among accepted attempts: {logical_error_rate:.4e}")
print(f"Logical error rate among accepted attempts: {logical_error_rate}")

In [ ]:
# after re.estimate_runtime()
ready_idx = re.ready_stage_idx

# choose one:
m_first = re.calculate_mean(record_type="first_reach")
m_until = re.calculate_mean(record_type="until_success")

# accumulated times at ready stage (mean over epochs)
print("FIRST_REACH @ ready")
print("decode :", m_first["decode"][ready_idx])
print("feedback:", m_first["feedback"][ready_idx])
print("gate   :", m_first["gate"][ready_idx])
print("total  :", m_first["total"][ready_idx])

print("UNTIL_SUCCESS @ ready")
print("decode :", m_until["decode"][ready_idx])
print("feedback:", m_until["feedback"][ready_idx])
print("gate   :", m_until["gate"][ready_idx])
print("total  :", m_until["total"][ready_idx])

##### With selection based on gap threshold in escape stage(complete)

In [ ]:
gap_check = True

re.estimate_runtime(gap_check=gap_check, partial_mask=False, selector=selector, left_len=left_len, top_len=top_len, t=t, mode=mode, stage_name="escape", decoder_time_measure_mode="parallel")

In [ ]:
print("decode calls:", len(re.decode_call_times_ns))
print("mean decode ns:", np.mean(re.decode_call_times_ns))
print("p50/p90 ns:", np.percentile(re.decode_call_times_ns, [50, 90]))

In [ ]:
display(re.plot_life_time_notebook())
avg_decoding_latency = re.report_average_decode_latency_ns()
logical_error_rate = re.logical_error_rate
# print(f"Logical error rate among accepted attempts: {logical_error_rate:.4e}")
print(f"Logical error rate among accepted attempts: {logical_error_rate}")

In [ ]:
# after re.estimate_runtime()
ready_idx = re.ready_stage_idx

# choose one:
m_first = re.calculate_mean(record_type="first_reach")
m_until = re.calculate_mean(record_type="until_success")

# accumulated times at ready stage (mean over epochs)
print("FIRST_REACH @ ready")
print("decode :", m_first["decode"][ready_idx])
print("feedback:", m_first["feedback"][ready_idx])
print("gate   :", m_first["gate"][ready_idx])
print("total  :", m_first["total"][ready_idx])

print("UNTIL_SUCCESS @ ready")
print("decode :", m_until["decode"][ready_idx])
print("feedback:", m_until["feedback"][ready_idx])
print("gate   :", m_until["gate"][ready_idx])
print("total  :", m_until["total"][ready_idx])

##### With selection based on gap threshold in escape stage(partial mask)

In [ ]:
gap_check = True

re.estimate_runtime(gap_check=gap_check, partial_mask=True, selector=selector, left_len=left_len, top_len=top_len, t=t, mode=mode, stage_name="escape", decoder_time_measure_mode="parallel")

In [ ]:
print("decode calls:", len(re.decode_call_times_ns))
print("mean decode ns:", np.mean(re.decode_call_times_ns))
print("p50/p90 ns:", np.percentile(re.decode_call_times_ns, [50, 90]))

In [ ]:
display(re.plot_life_time_notebook())
avg_decoding_latency = re.report_average_decode_latency_ns()
logical_error_rate = re.logical_error_rate
# print(f"Logical error rate among accepted attempts: {logical_error_rate:.4e}")
print(f"Logical error rate among accepted attempts: {logical_error_rate}")

In [ ]:
# after re.estimate_runtime()
ready_idx = re.ready_stage_idx

# choose one:
m_first = re.calculate_mean(record_type="first_reach")
m_until = re.calculate_mean(record_type="until_success")

# accumulated times at ready stage (mean over epochs)
print("FIRST_REACH @ ready")
print("decode :", m_first["decode"][ready_idx])
print("feedback:", m_first["feedback"][ready_idx])
print("gate   :", m_first["gate"][ready_idx])
print("total  :", m_first["total"][ready_idx])

print("UNTIL_SUCCESS @ ready")
print("decode :", m_until["decode"][ready_idx])
print("feedback:", m_until["feedback"][ready_idx])
print("gate   :", m_until["gate"][ready_idx])
print("total  :", m_until["total"][ready_idx])

In [ ]:
# d = 3 color code

# Generate the circuit
basis = 'Y'
gateset = 'css'
circuit_type = circuit_types[4]  # 'end2end-inplace-distillation'
noise_model = noise_models[1]  # 'circuit-level-SI1000'
noise_strength = 0.001
injection_protocol = injection_protocols[2]  # 'unitary'
r_in_escape = 3   # r1
r_post_escape = 10 # r2
d1 = 3
d2 = 15
HasNoise = True

epoch_2 = 100000

feedback_time_ns = 2000
manual_avg_decoder_latency_ns = 10000
use_measured_decode_time = True


re_2 = re_mod.RuntimeEstimator(
    epoch=epoch_2,                    # how many successful outputs to emulate
    basis=basis,
    gateset=gateset,
    circuit_type=circuit_type,
    noise_model=noise_model,
    noise_strength=noise_strength,
    distance=d1,                   # or 5
    d2=d2,
    feedback_time_ns=feedback_time_ns,
    decoder_latency_ns=manual_avg_decoder_latency_ns,  # 10 us
    use_measured_decode_time=use_measured_decode_time,
    wait_rounds_for_plot=r_post_escape,      # plot expansion only
)

In [ ]:
gap_check_2 = True

re_2.estimate_runtime(gap_check=gap_check_2, partial_mask=True, selector=selector, left_len=left_len, top_len=top_len, t=t, mode=mode, stage_name="escape", decoder_time_measure_mode="parallel")

In [ ]:
display(re_2.plot_life_time_notebook())
avg_decoding_latency_2 = re_2.report_average_decode_latency_ns()
logical_error_rate_2 = re_2.logical_error_rate
# print(f"Logical error rate among accepted attempts: {logical_error_rate:.4e}")
print(f"Logical error rate among accepted attempts: {logical_error_rate_2}")

In [ ]:
# after re.estimate_runtime()
ready_idx_2 = re_2.ready_stage_idx

# choose one:
m_first_2 = re_2.calculate_mean(record_type="first_reach")
m_until_2 = re_2.calculate_mean(record_type="until_success")

# accumulated times at ready stage (mean over epochs)
print("FIRST_REACH @ ready")
print("decode :", m_first_2["decode"][ready_idx_2])
print("feedback:", m_first_2["feedback"][ready_idx_2])
print("gate   :", m_first_2["gate"][ready_idx_2])
print("total  :", m_first_2["total"][ready_idx_2])

print("UNTIL_SUCCESS @ ready")
print("decode :", m_until_2["decode"][ready_idx_2])
print("feedback:", m_until_2["feedback"][ready_idx_2])
print("gate   :", m_until_2["gate"][ready_idx_2])
print("total  :", m_until_2["total"][ready_idx_2])

#### Follow assumption: decoding = 10us, feedback = 1us

In [ ]:
# d = 3 color code

# Generate the circuit
basis = 'Y'
gateset = 'css'
circuit_type = circuit_types[4]  # 'end2end-inplace-distillation'
noise_model = noise_models[1]  # 'circuit-level-SI1000'
noise_strength = 0.001
injection_protocol = injection_protocols[2]  # 'unitary'
r_in_escape = 3   # r1
r_post_escape = 10 # r2
d1 = 3
d2 = 15
HasNoise = True

epoch = 1000

feedback_time_ns = 1000
manual_avg_decoder_latency_ns = 10000
use_measured_decode_time = False


re = re_mod.RuntimeEstimator(
    epoch=epoch,                    # how many successful outputs to emulate
    basis=basis,
    gateset=gateset,
    circuit_type=circuit_type,
    noise_model=noise_model,
    noise_strength=noise_strength,
    distance=d1,                   # or 5
    d2=d2,
    feedback_time_ns=feedback_time_ns,
    decoder_latency_ns=manual_avg_decoder_latency_ns,  # 10 us
    use_measured_decode_time=use_measured_decode_time,
    wait_rounds_for_plot=r_post_escape,      # plot expansion only
)

In [ ]:
re.estimate_runtime()

In [ ]:
display(re.plot_life_time_notebook())
avg_decoding_latency = re.report_average_decode_latency_ns()

In [ ]:
# after re.estimate_runtime()
ready_idx = re.ready_stage_idx

# choose one:
m_first = re.calculate_mean(record_type="first_reach")
m_until = re.calculate_mean(record_type="until_success")

# accumulated times at ready stage (mean over epochs)
print("FIRST_REACH @ ready")
print("decode :", m_first["decode"][ready_idx])
print("feedback:", m_first["feedback"][ready_idx])
print("gate   :", m_first["gate"][ready_idx])
print("total  :", m_first["total"][ready_idx])

print("UNTIL_SUCCESS @ ready")
print("decode :", m_until["decode"][ready_idx])
print("feedback:", m_until["feedback"][ready_idx])
print("gate   :", m_until["gate"][ready_idx])
print("total  :", m_until["total"][ready_idx])

#### Follow Assumption: decoding = 1us, feedback = 500ns

In [ ]:
# d = 3 color code

# Generate the circuit
basis = 'Y'
gateset = 'css'
circuit_type = circuit_types[4]  # 'end2end-inplace-distillation'
noise_model = noise_models[1]  # 'circuit-level-SI1000'
noise_strength = 0.001
injection_protocol = injection_protocols[2]  # 'unitary'
r_in_escape = 3   # r1
r_post_escape = 10 # r2
d1 = 3
d2 = 15
HasNoise = True

epoch = 1000

feedback_time_ns = 500
manual_avg_decoder_latency_ns = 1000

use_measured_decode_time = False


re = re_mod.RuntimeEstimator(
    epoch=epoch,                    # how many successful outputs to emulate
    basis=basis,
    gateset=gateset,
    circuit_type=circuit_type,
    noise_model=noise_model,
    noise_strength=noise_strength,
    distance=d1,                   # or 5
    d2=d2,
    feedback_time_ns=feedback_time_ns,
    decoder_latency_ns=manual_avg_decoder_latency_ns,  # 10 us
    use_measured_decode_time=use_measured_decode_time,
    wait_rounds_for_plot=r_post_escape,      # plot expansion only
)

In [ ]:
re.estimate_runtime()

In [ ]:
display(re.plot_life_time_notebook())
avg_decoding_latency = re.report_average_decode_latency_ns()

In [ ]:
# after re.estimate_runtime()
ready_idx = re.ready_stage_idx

# choose one:
m_first = re.calculate_mean(record_type="first_reach")
m_until = re.calculate_mean(record_type="until_success")

# accumulated times at ready stage (mean over epochs)
print("FIRST_REACH @ ready")
print("decode :", m_first["decode"][ready_idx])
print("feedback:", m_first["feedback"][ready_idx])
print("gate   :", m_first["gate"][ready_idx])
print("total  :", m_first["total"][ready_idx])

print("UNTIL_SUCCESS @ ready")
print("decode :", m_until["decode"][ready_idx])
print("feedback:", m_until["feedback"][ready_idx])
print("gate   :", m_until["gate"][ready_idx])
print("total  :", m_until["total"][ready_idx])

### Distance 5 color code

In [ ]:
# d = 5 color code

# Generate the circuit
basis = 'Y'
gateset = 'css'
circuit_type = circuit_types[4]  # 'end2end-inplace-distillation'
noise_model = noise_models[1]  # 'circuit-level-SI1000'
noise_strength = 0.001
injection_protocol = injection_protocols[2]  # 'unitary'
r_in_escape = 5   # r1
r_post_escape = 10 # r2
d1 = 5
d2 = 15
HasNoise = True

epoch = 1000

feedback_time_ns = 2000
manual_avg_decoder_latency_ns = 10000
use_measured_decode_time = True


re = re_mod.RuntimeEstimator(
    epoch=epoch,                    # how many successful outputs to emulate
    basis=basis,
    gateset=gateset,
    circuit_type=circuit_type,
    noise_model=noise_model,
    noise_strength=noise_strength,
    distance=d1,                   # or 5
    d2=d2,
    feedback_time_ns=feedback_time_ns,
    decoder_latency_ns=manual_avg_decoder_latency_ns,  # 10 us
    use_measured_decode_time=use_measured_decode_time,
    wait_rounds_for_plot=r_post_escape,      # plot expansion only
)

In [ ]:
re.estimate_runtime()

In [ ]:
display(re.plot_life_time_notebook())
avg_decoding_latency = re.report_average_decode_latency_ns()

In [ ]:
# after re.estimate_runtime()
ready_idx = re.ready_stage_idx

# choose one:
m_first = re.calculate_mean(record_type="first_reach")
m_until = re.calculate_mean(record_type="until_success")

# accumulated times at ready stage (mean over epochs)
print("FIRST_REACH @ ready")
print("decode :", m_first["decode"][ready_idx])
print("feedback:", m_first["feedback"][ready_idx])
print("gate   :", m_first["gate"][ready_idx])
print("total  :", m_first["total"][ready_idx])

print("UNTIL_SUCCESS @ ready")
print("decode :", m_until["decode"][ready_idx])
print("feedback:", m_until["feedback"][ready_idx])
print("gate   :", m_until["gate"][ready_idx])
print("total  :", m_until["total"][ready_idx])

#### Follow Assumption: decoding = 10us, feedback = 1us

In [ ]:
# d = 5 color code

# Generate the circuit
basis = 'Y'
gateset = 'css'
circuit_type = circuit_types[4]  # 'end2end-inplace-distillation'
noise_model = noise_models[1]  # 'circuit-level-SI1000'
noise_strength = 0.001
injection_protocol = injection_protocols[2]  # 'unitary'
r_in_escape = 5   # r1
r_post_escape = 10 # r2
d1 = 5
d2 = 15
HasNoise = True

epoch = 1000

feedback_time_ns = 1000
manual_avg_decoder_latency_ns = 10000
use_measured_decode_time = False


re = re_mod.RuntimeEstimator(
    epoch=epoch,                    # how many successful outputs to emulate
    basis=basis,
    gateset=gateset,
    circuit_type=circuit_type,
    noise_model=noise_model,
    noise_strength=noise_strength,
    distance=d1,                   # or 5
    d2=d2,
    feedback_time_ns=feedback_time_ns,
    decoder_latency_ns=manual_avg_decoder_latency_ns,  # 10 us
    use_measured_decode_time=use_measured_decode_time,
    wait_rounds_for_plot=r_post_escape,      # plot expansion only
)

re.estimate_runtime()

In [ ]:
display(re.plot_life_time_notebook())
avg_decoding_latency = re.report_average_decode_latency_ns()

In [ ]:
# after re.estimate_runtime()
ready_idx = re.ready_stage_idx

# choose one:
m_first = re.calculate_mean(record_type="first_reach")
m_until = re.calculate_mean(record_type="until_success")

# accumulated times at ready stage (mean over epochs)
print("FIRST_REACH @ ready")
print("decode :", m_first["decode"][ready_idx])
print("feedback:", m_first["feedback"][ready_idx])
print("gate   :", m_first["gate"][ready_idx])
print("total  :", m_first["total"][ready_idx])

print("UNTIL_SUCCESS @ ready")
print("decode :", m_until["decode"][ready_idx])
print("feedback:", m_until["feedback"][ready_idx])
print("gate   :", m_until["gate"][ready_idx])
print("total  :", m_until["total"][ready_idx])

#### Follow Assumption: decoding = 1us, feedback = 500ns

In [ ]:
# d = 5 color code

# Generate the circuit
basis = 'Y'
gateset = 'css'
circuit_type = circuit_types[4]  # 'end2end-inplace-distillation'
noise_model = noise_models[1]  # 'circuit-level-SI1000'
noise_strength = 0.001
injection_protocol = injection_protocols[2]  # 'unitary'
r_in_escape = 5   # r1
r_post_escape = 10 # r2
d1 = 5
d2 = 15
HasNoise = True

epoch = 1000

feedback_time_ns = 500
manual_avg_decoder_latency_ns = 1000
use_measured_decode_time = False


re = re_mod.RuntimeEstimator(
    epoch=epoch,                    # how many successful outputs to emulate
    basis=basis,
    gateset=gateset,
    circuit_type=circuit_type,
    noise_model=noise_model,
    noise_strength=noise_strength,
    distance=d1,                   # or 5
    d2=d2,
    feedback_time_ns=feedback_time_ns,
    decoder_latency_ns=manual_avg_decoder_latency_ns,  # 10 us
    use_measured_decode_time=use_measured_decode_time,
    wait_rounds_for_plot=r_post_escape,      # plot expansion only
)

re.estimate_runtime()

In [ ]:
display(re.plot_life_time_notebook())
avg_decoding_latency = re.report_average_decode_latency_ns()

In [ ]:
# after re.estimate_runtime()
ready_idx = re.ready_stage_idx

# choose one:
m_first = re.calculate_mean(record_type="first_reach")
m_until = re.calculate_mean(record_type="until_success")

# accumulated times at ready stage (mean over epochs)
print("FIRST_REACH @ ready")
print("decode :", m_first["decode"][ready_idx])
print("feedback:", m_first["feedback"][ready_idx])
print("gate   :", m_first["gate"][ready_idx])
print("total  :", m_first["total"][ready_idx])

print("UNTIL_SUCCESS @ ready")
print("decode :", m_until["decode"][ready_idx])
print("feedback:", m_until["feedback"][ready_idx])
print("gate   :", m_until["gate"][ready_idx])
print("total  :", m_until["total"][ready_idx])

## Test the detector selection

In [ ]:
# Generate the circuit
basis = 'Y'
gateset = 'css'
circuit_type = circuit_types[4]  # 'end2end-inplace-distillation'
noise_model = noise_models[1]  # 'uniform-depolarizing'
noise_strength = 0.001
injection_protocol = injection_protocols[2]  # 'unitary'
d1 = 3
d2 = 15
r_in_escape = d1   # r1
r_post_escape = 0  # r2

cg = make_tsim_cg(
    basis=basis,
    gateset=gateset,
    circuit_type=circuit_type,
    noise_model=noise_model,
    noise_strength=noise_strength,
    injection_protocol=injection_protocol,
    r_in_escape=r_in_escape,
    d1=d1,
    r_post_escape=r_post_escape,
    d2=d2,
)

In [ ]:
selector_check = DetectorSelection3D(circuit_generator=cg)

In [ ]:
# Build mask (rectangle)
left_len = 7
top_len = 6
t = 4
# mode = 'rectangle'   # or 'triangle'
mode = 'triangle'

mask_bool, mask_packed = selector_check.build_color_region_mask(
    left_len=left_len,
    top_len=top_len,
    t=t,
    mode=mode,   # or 'triangle'
    stage_name='escape'
)

In [ ]:
# Inline visualization in notebook
selector_check.visualize_selected_region_plot()

In [ ]:
left_len_start = 1
left_len_end = 12
top_len_start = 8
top_len_end = 12
t_2 = 5
mode_2 = 'rectangle'
triangle_half = "left"

mask_bool_check, mask_packed_check, meta_check = selector_check.build_partial_region_apart_color_mask(
    left_len_start=left_len_start, left_len_end=left_len_end,
    top_len_start=top_len_start, top_len_end=top_len_end,
    t=t_2,
    mode=mode_2,          # or "rectangle"
    triangle_half=triangle_half,    # "left" or "right"
    stage_name="escape",
)

In [ ]:
selector_check.visualize_partial_region_apart_color_plot(mask_bool=mask_bool_check, metadata=meta_check)

In [ ]:
masks_bool, masks_packed, meta = selector_check.build_partial_region_multi_mask(
    left_len=7, top_len=8, t=5, mode="triangle", stage_name="escape"
)

In [ ]:
selector_check.visualize_partial_region_multi_plot(masks_bool=masks_bool, metadata=meta)

## Test the Gap estimation with partial detector selection

#### Distance 3 Color code

In [ ]:
# Generate the circuit
basis = 'Y'
gateset = 'css'
circuit_type = circuit_types[4]  # 'end2end-inplace-distillation'
noise_model = noise_models[1]  # 'uniform-depolarizing'
noise_strength = 0.001
injection_protocol = injection_protocols[2]  # 'unitary'
d1 = 3
d2 = 15
r_in_escape = d1   # r1
r_post_escape = 0  # r2

cg = make_tsim_cg(
    basis=basis,
    gateset=gateset,
    circuit_type=circuit_type,
    noise_model=noise_model,
    noise_strength=noise_strength,
    injection_protocol=injection_protocol,
    r_in_escape=r_in_escape,
    d1=d1,
    r_post_escape=r_post_escape,
    d2=d2,
)

In [ ]:
gap_gen = GapThreshotGenerate(
        basis=basis,
        gateset=gateset,
        circuit_type=circuit_type,
        noise_model=noise_model,
        noise_strength=noise_strength,
        distance=d1,
        d2=d2,
        injection_protocol=injection_protocol,
        r_in_escape=r_in_escape,
        threshold_shots=1024 * 100
    )

print(gap_gen.gap_threshold)

In [ ]:
selector = DetectorSelection3D(circuit_generator=cg)

In [ ]:
# Build mask (rectangle)
left_len = 7
top_len = 6
t = 4
# mode = 'rectangle'   # or 'triangle'
mode = 'triangle'

mask_bool, mask_packed = selector.build_color_region_mask(
    left_len=left_len,
    top_len=top_len,
    t=t,
    mode=mode,   # or 'triangle'
    stage_name='escape'
)

print(selector.selection_metadata)
print(mask_bool.shape, mask_packed.shape, mask_bool.sum())

In [ ]:
# Inline visualization in notebook
selector.visualize_selected_region_plot()

In [ ]:
gap_col = GapSensitivityCollect(circuit_generator=cg)

In [ ]:
gap_average_parameters = [0.9, 0.1, 0, 0, 0]
left_len_avg = 8
top_len_avg = 7
t_avg = 5
mode_avg = "triangle"
check_bescombo = False

gap_col.collect_partial_detector_gap_sens(
    shots=1000,
    selector=selector,
    left_lengths=(left_len, left_len + 4),   # (start, end)
    top_lengths=(top_len, top_len + 4),    # (start, end)
    t_lengths=(t, t + 4),      # (start, end)
    mode='triangle',      # or 'triangle'
    stage_name='escape',   # passed to selector.build_color_region_mask
    check_bestcombo=check_bescombo,
    left_len_avg=left_len_avg,
    top_len_avg=top_len_avg,
    t_avg=t_avg,
    mode_avg=mode_avg
)

In [ ]:
gap_threshold = gap_gen.gap_threshold
gap_threshold = 66

gap_col.collect_partial_accpet_and_reject(gap_threshold=gap_threshold, gap_threshold_pure_avg=66)

In [ ]:
# 3) Compute and store best-combo distribution
gap_col.calculate_partial_gap_best_combo_distribution()
# 4) Print summary (without sample lists)
_ = gap_col.print_partial_gap_best_combo_distribution()

In [ ]:
display(gap_col.false_accept_reject_detail_plot(select_type="weighted_regions", first_shots_num=10))

In [ ]:
display(gap_col.false_accept_reject_detail_plot(select_type="pure_avg", first_shots_num=10))

In [ ]:
shot_idx = 26
print("partial gap: ", gap_col.partial_gap_sens_data["closest_matches"][shot_idx]["partial_gap"])
print("complete gap: ", gap_col.partial_gap_sens_data["closest_matches"][shot_idx]["complete_gap"])
print("multi mask gap 0: ", gap_col.partial_gap_sens_data["closest_matches"][shot_idx]["weighted_region_gaps"][0])
print("multi mask gap 1: ", gap_col.partial_gap_sens_data["closest_matches"][shot_idx]["weighted_region_gaps"][1])
print("multi mask gap 2: ", gap_col.partial_gap_sens_data["closest_matches"][shot_idx]["weighted_region_gaps"][2])
print("multi mask gap 3: ", gap_col.partial_gap_sens_data["closest_matches"][shot_idx]["weighted_region_gaps"][3])
print("multi mask gap 4: ", gap_col.partial_gap_sens_data["closest_matches"][shot_idx]["weighted_region_gaps"][4])
print("weighted gap: ", gap_col.partial_gap_sens_data["closest_matches"][shot_idx]["weighted_avg_gap"])
print("pure average gap: ", gap_col.partial_gap_sens_data["closest_matches"][shot_idx]["pure_avg_gap"])

In [ ]:
# Quick check
gap_col.partial_gap_sens_data["remained_shots"], len(gap_col.partial_gap_sens_data["combos"])

In [ ]:
# 3D plot for one kept shot index
gap_col.partial_detector_gap_sens_svg(shot_idx=0)

In [ ]:
gap_col.print_partial_gap_match_for_shot(0)

In [ ]:
gap_col.print_partial_gap_match_for_shot(9)

In [ ]:
left_len_check = 7
top_len_check = 10
t_check = 5
mode_check = 'triangle'   # or 'triangle'

mask_bool_check, mask_packed_check = selector.build_color_region_mask(
    left_len=left_len_check,
    top_len=top_len_check,
    t=t_check,
    mode=mode_check,   # or 'triangle'
    stage_name='escape'
)

selector.visualize_selected_region_plot()

In [ ]:
# 3) Compute and store best-combo distribution
gap_col.calculate_partial_gap_best_combo_distribution()

In [ ]:
# 4) Print summary (without sample lists)
_ = gap_col.print_partial_gap_best_combo_distribution()

In [ ]:
display(gap_col.complete_gap_vs_pure_avg_gap_plot())

In [ ]:
# 5) Plot in notebook
gap_col.partial_gap_best_combo_distribution_notebook(value_type="left_len")

In [ ]:
gap_col.partial_gap_best_combo_distribution_notebook(value_type="top_len")

In [ ]:
gap_col.partial_gap_best_combo_distribution_notebook(value_type="t")

In [ ]:
gap_col.partial_gap_best_combo_distribution_notebook(value_type="abs_diff")

#### Distance 5 Color code

In [ ]:
# Generate the circuit
basis = 'Y'
gateset = 'css'
circuit_type = circuit_types[4]  # 'end2end-inplace-distillation'
noise_model = noise_models[1]  # 'uniform-depolarizing'
noise_strength = 0.001
injection_protocol = injection_protocols[2]  # 'unitary'
d1 = 5
d2 = 15
r_in_escape = d1   # r1
r_post_escape = 0  # r2

cg = make_tsim_cg(
    basis=basis,
    gateset=gateset,
    circuit_type=circuit_type,
    noise_model=noise_model,
    noise_strength=noise_strength,
    injection_protocol=injection_protocol,
    r_in_escape=r_in_escape,
    d1=d1,
    r_post_escape=r_post_escape,
    d2=d2,
)

In [ ]:
gap_gen = GapThreshotGenerate(
        basis=basis,
        gateset=gateset,
        circuit_type=circuit_type,
        noise_model=noise_model,
        noise_strength=noise_strength,
        distance=d1,
        d2=d2,
        injection_protocol=injection_protocol,
        r_in_escape=r_in_escape,
        threshold_shots=1024 * 100
    )

print(gap_gen.gap_threshold)

In [ ]:
selector = DetectorSelection3D(circuit_generator=cg)

In [ ]:
# Build mask (rectangle)
left_len = 8
top_len = 8
t = 10
mode = 'triangle'   # or 'triangle'

mask_bool, mask_packed = selector.build_color_region_mask(
    left_len=left_len,
    top_len=top_len,
    t=t,
    mode=mode,   # or 'triangle'
    stage_name='escape'
)

print(selector.selection_metadata)
print(mask_bool.shape, mask_packed.shape, mask_bool.sum())

In [ ]:
# Inline visualization in notebook
selector.visualize_selected_region_plot()

In [ ]:
left_len = 8
top_len = 8
t = 10


diff_left_len = 10
diff_top_len = 10
diff_t = 4

In [ ]:
# Build mask (rectangle)
mode = 'triangle'   # or 'triangle'

mask_bool, mask_packed = selector.build_color_region_mask(
    left_len=left_len+diff_left_len,
    top_len=top_len+diff_top_len,
    t=t+diff_t,
    mode=mode,   # or 'triangle'
    stage_name='escape'
)

In [ ]:
# Inline visualization in notebook
selector.visualize_selected_region_plot()

In [ ]:
gap_col = GapSensitivityCollect(circuit_generator=cg)

In [ ]:
# diff_left_len = 15
# diff_top_len = 10
# diff_t = 4
gap_threshold = gap_gen.gap_threshold
gap_col.collect_partial_detector_gap_sens(
    shots=40000,
    selector=selector,
    left_lengths=(left_len, left_len + diff_left_len),   # (start, end)
    top_lengths=(top_len, top_len + diff_top_len),    # (start, end)
    t_lengths=(t, t + diff_t),      # (start, end)
    mode='triangle',      # or 'triangle'
    stage_name='escape',   # passed to selector.build_color_region_mask
    gap_threshold = gap_threshold
)

In [ ]:
# Quick check
gap_col.partial_gap_sens_data["remained_shots"], len(gap_col.partial_gap_sens_data["combos"])

In [ ]:
# 3D plot for one kept shot index
gap_col.partial_detector_gap_sens_svg(shot_idx=0)

In [ ]:
gap_col.print_partial_gap_match_for_shot(0)

In [ ]:
left_len_check = 10
top_len_check = 12
t_check = 12
mode_check = 'triangle'   # or 'triangle'

mask_bool_check, mask_packed_check = selector.build_color_region_mask(
    left_len=left_len_check,
    top_len=top_len_check,
    t=t_check,
    mode=mode_check,   # or 'triangle'
    stage_name='escape'
)

selector.visualize_selected_region_plot()

In [ ]:
# 3) Compute and store best-combo distribution
gap_col.calculate_partial_gap_best_combo_distribution()

In [ ]:
# 4) Print summary (without sample lists)
_ = gap_col.print_partial_gap_best_combo_distribution()

In [ ]:
# 5) Plot in notebook
gap_col.partial_gap_best_combo_distribution_notebook(value_type="left_len")

In [ ]:
gap_col.partial_gap_best_combo_distribution_notebook(value_type="top_len")

In [ ]:
gap_col.partial_gap_best_combo_distribution_notebook(value_type="t")

In [ ]:
gap_col.partial_gap_best_combo_distribution_notebook(value_type="abs_diff")